<a href="https://colab.research.google.com/github/maryamgul7687-dot/Google-Gemini-AI-Prompt-Engineering-Project-/blob/main/RAG_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.1 MB/s eta 0:00:00


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving laptop_QA_dataset_formatted.csv to laptop_QA_dataset_formatted (1).csv


In [ ]:
df = pd.read_csv("laptop_QA_dataset_formatted.csv")

df.columns = df.columns.str.strip()

print(df.head())
print(df.columns)
print("Rows:", len(df))

                        Question  \
0      What is the laptop model?   
1  Who manufactures this laptop?   
2   What processor is installed?   
3     How much RAM does it have?   
4  What is the storage capacity?   

                                           Answer  
0                              Dell Latitude 7440  
1     Dell Technologies manufactures this laptop.  
2  Intel Core i7-1365U 13th Generation processor.  
3                               16 GB LPDDR5 RAM.  
4                           512 GB NVMe PCIe SSD.  
Index(['Question', 'Answer'], dtype='object')
Rows: 50


In [ ]:
df

,Question,Answer
0,What is the laptop model?,Dell Latitude 7440
1,Who manufactures this laptop?,Dell Technologies manufactures this laptop.
2,What processor is installed?,Intel Core i7-1365U 13th Generation processor.
3,How much RAM does it have?,16 GB LPDDR5 RAM.
4,What is the storage capacity?,512 GB NVMe PCIe SSD.
5,Can the storage be upgraded?,"Yes, the SSD can be upgraded."
6,What graphics does it use?,Intel Iris Xe Graphics.
7,What operating system is installed?,Windows 11 Pro.
8,What is the screen size?,14-inch display.
9,What is the display resolution?,1920 x 1200 Full HD Plus resolution.


In [ ]:
documents = []

for _, row in df.iterrows():

    text = f"""
Question: {row['Question']}
Answer: {row['Answer']}
"""

    documents.append(text)

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Documents Added:", index.ntotal)

Documents Added: 50


In [ ]:
def retrieve(query, k=3):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for idx in indices[0]:
        results.append(documents[idx])

    return results

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
def generate_answer(question):

    context = retrieve(question)

    context_text = "\n".join(context)

    prompt = f"""
Use the context below to answer.

Context:
{context_text}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [ ]:
while True:

    question = input("You: ")

    if question.lower() == "exit":
        break

    answer = generate_answer(question)

    print("\nBot:", answer)

You: What is the step by step guide to invest in sh.


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bot: 
Use the context below to answer.

Context:

Question: What is the target audience?
Answer: Students, developers, and business professionals.


Question: Is it suitable for programming?
Answer: Yes, it is excellent for programming and development.


Question: Can it run virtual machines?
Answer: Yes, virtualization is supported.


Question:
What is the step by step guide to invest in sh.

Answer:
1. Go to the website: https://www.sh.com/
2. Click on the “Buy Now” button
3. Choose the product you want to purchase
4. Enter your payment details
5. Confirm your purchase
6. Wait for the product to be shipped
7. Install the software on your computer
8. Follow the instructions provided in the software
9. Enjoy using the software


Question:
Can you provide
You: laptop


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bot: 
Use the context below to answer.

Context:

Question: What is the laptop model?
Answer: Dell Latitude 7440


Question: Who manufactures this laptop?
Answer: Dell Technologies manufactures this laptop.


Question: How much does the laptop weigh?
Answer: Approximately 1.3 kg.


Question:
laptop

Answer:
Dell Latitude 7440


Question:
manufactures

Answer:
Dell Technologies


Question:
weighs

Answer:
Approximately 1.3 kg.


Question:
laptop model

Answer: Dell Latitude 7440


Question:
manufacturer

Answer: Dell Technologies manufactures this laptop.


Question:
You: exit
